# Supplementary Figure S5: Full Stratified Robustness

- Figure / panels: `FigS5a-d` stratified robustness, `FigS5e` split-wise global structure recovery
- Inputs: metrics and payloads for the selected datasets
- Outputs: `artifacts/paper_figures/supp/FigS5_Robustness/*`
- Run order: optional; run after Fig3 if the full robustness supplement is needed
- Dataset selector: edit `SELECTED_DATASET_KEYS` in the parameter cell below
- Role: Supplementary


In [ ]:
from __future__ import annotations

import importlib
import sys
import shutil
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style
import scripts.trishift.analysis.split_structure_recovery as split_structure_recovery
import scripts.trishift.analysis.stratified_benchmark as stratified_benchmark

apply_gears_paper_style(font_scale=1.0)
importlib.reload(stratified_benchmark)
importlib.reload(split_structure_recovery)



In [ ]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit'},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson'},
    {'dataset_key': 'norman', 'dataset_label': 'Norman'},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562'},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1'},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['adamson', 'norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

MODELS = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt']  # edit here
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]

STRUCTURE_PANEL_ENABLED = True  # edit here
STRUCTURE_PANEL_DATASET = 'norman'  # edit here
STRUCTURE_PANEL_MODELS = ['trishift_nearest']  # edit here
STRUCTURE_PANEL_MAX_CELLS_PER_CONDITION = 28
STRUCTURE_PANEL_CLUSTER_K = 4

OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS5_Robustness'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
summary_frames = []
print('Selected datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Selected models:', MODELS)
for spec in DATASET_SPECS:
    try:
        result = stratified_benchmark.run_stratified_benchmark(dataset=spec['dataset_key'], models=MODELS, split_ids=SPLIT_IDS, out_root=OUT_ROOT / spec['dataset_key'], paths_path=repo_root / 'configs' / 'paths.yaml')
    except Exception:
        continue
    df = result['summary_df'].copy(); df['dataset_label'] = spec['dataset_label']; summary_frames.append(df)
summary_df = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
summary_df.to_csv(OUT_ROOT / 'figs5_summary_all.csv', index=False, encoding='utf-8-sig')
display(summary_df.head())

if STRUCTURE_PANEL_ENABLED:
    structure_spec = next((spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] == STRUCTURE_PANEL_DATASET), None)
    if structure_spec is None:
        raise ValueError(f'Unknown STRUCTURE_PANEL_DATASET: {STRUCTURE_PANEL_DATASET}')
    if STRUCTURE_PANEL_DATASET not in SELECTED_DATASET_KEYS:
        print(f'Skip structure panel because {STRUCTURE_PANEL_DATASET!r} is not in SELECTED_DATASET_KEYS')
    else:
        structure_result = split_structure_recovery.run_split_structure_recovery(
            dataset=STRUCTURE_PANEL_DATASET,
            dataset_label=structure_spec['dataset_label'],
            models=STRUCTURE_PANEL_MODELS,
            split_ids=SPLIT_IDS,
            out_root=OUT_ROOT,
            paths_path=repo_root / 'configs' / 'paths.yaml',
            max_cells_per_condition=STRUCTURE_PANEL_MAX_CELLS_PER_CONDITION,
            cluster_k=STRUCTURE_PANEL_CLUSTER_K,
        )
        print(structure_result.figure_path)
        if structure_result.figure_path.exists():
            display(Image(filename=str(structure_result.figure_path), width=1200))
        display(structure_result.summary_df.head())


standard_panels = [
    ('adamson/difficulty_scatter.png', 'figs5a_adamson_difficulty_scatter.png'),
    ('adamson/stratified_summary_barplot.png', 'figs5b_adamson_stratified_summary.png'),
    ('norman/difficulty_scatter.png', 'figs5c_norman_difficulty_scatter.png'),
    ('norman/stratified_summary_barplot.png', 'figs5d_norman_stratified_summary.png'),
]
for rel_src, out_name in standard_panels:
    src_path = OUT_ROOT / rel_src
    dst_path = OUT_ROOT / out_name
    if src_path.exists():
        shutil.copy2(src_path, dst_path)
        print(f'Exported {dst_path.name}')


# Combined Adamson robustness figure: scatter + stratified summary + train-distance trend
adamson_scatter = OUT_ROOT / 'adamson' / 'difficulty_scatter.csv'
adamson_summary = OUT_ROOT / 'adamson' / 'stratified_summary.csv'
combined_out = OUT_ROOT / 'figs5ab_adamson_robustness_combined.png'
line_out = OUT_ROOT / 'figs5c_adamson_train_distance_trend.png'
if adamson_scatter.exists() and adamson_summary.exists():
    scatter_df = pd.read_csv(adamson_scatter)
    summary_plot_df = pd.read_csv(adamson_summary)
    model_order = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt']
    model_labels = {
        'trishift_nearest': 'TriShift',
        'biolord': 'biolord',
        'gears': 'GEARS',
        'genepert': 'GenePert',
        'scgpt': 'scGPT',
    }
    model_colors = {
        'TriShift': '#4C78A8',
        'biolord': '#0073C2',
        'GEARS': '#D04E59',
        'GenePert': '#54A24B',
        'scGPT': '#B279A2',
    }
    scatter_df = scatter_df[scatter_df['model_name'].isin(model_order)].copy()
    scatter_df['display_name'] = scatter_df['model_name'].map(model_labels)
    summary_plot_df = summary_plot_df[summary_plot_df['model_name'].isin(model_order)].copy()
    summary_plot_df['display_name'] = summary_plot_df['model_name'].map(model_labels)

    fig = plt.figure(figsize=(16.0, 9.6), dpi=220)
    gs = fig.add_gridspec(2, 4, height_ratios=[1.18, 1.0], hspace=0.32, wspace=0.16)
    ax_scatter = fig.add_subplot(gs[0, :])
    sns.scatterplot(
        data=scatter_df,
        x='train_test_distance', y='pearson',
        hue='display_name',
        hue_order=[model_labels[m] for m in model_order],
        palette=model_colors,
        s=26, alpha=0.72, linewidth=0.0, ax=ax_scatter, legend=False,
    )
    ax_scatter.set_title('Difficulty vs performance', pad=10)
    ax_scatter.set_xlabel('Train-test distance')
    ax_scatter.set_ylabel('Pearson')
    ax_scatter.grid(axis='both', alpha=0.22)
    ax_scatter.text(-0.04, 1.03, 'a', transform=ax_scatter.transAxes, fontsize=18, fontweight='bold')

    stratum_specs = [
        ('effect_strength_bin', 'effect strength bin', ['weak', 'medium', 'strong']),
        ('train_distance_bin', 'train distance bin', ['near', 'medium', 'far']),
        ('deg_difficulty_bin', 'deg difficulty bin', ['easy', 'medium', 'hard']),
    ]
    bar_axes = [fig.add_subplot(gs[1, i]) for i in range(3)]
    for ax, (stratum_name, title, order) in zip(bar_axes, stratum_specs):
        work = summary_plot_df[summary_plot_df['stratum_name'] == stratum_name].copy()
        work = work.dropna(subset=['pearson'])
        if work.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue
        sns.barplot(
            data=work,
            x='stratum_value', y='pearson', hue='display_name',
            order=order,
            hue_order=[model_labels[m] for m in model_order],
            palette=model_colors,
            errorbar=None, saturation=0.95, ax=ax,
        )
        if ax.legend_ is not None:
            ax.legend_.remove()
        ax.set_title(title, fontsize=11, pad=6)
        ax.set_xlabel('')
        ax.set_ylabel('Pearson' if ax is bar_axes[0] else '')
        ax.set_ylim(0, 1.06)
        ax.grid(axis='y', alpha=0.20)
        for patch in ax.patches:
            patch.set_edgecolor('black')
            patch.set_linewidth(0.35)
        for p in ax.patches:
            h = p.get_height()
            if pd.notna(h):
                ax.text(p.get_x() + p.get_width()/2, min(h - 0.07, 1.02), f'{h:.2f}', ha='center', va='top', fontsize=7)
    bar_axes[0].text(-0.18, 1.08, 'b', transform=bar_axes[0].transAxes, fontsize=18, fontweight='bold')

    ax_line = fig.add_subplot(gs[1, 3])
    line_df = summary_plot_df[summary_plot_df['stratum_name'] == 'train_distance_bin'].copy()
    line_df['stratum_value'] = pd.Categorical(line_df['stratum_value'], categories=['near', 'medium', 'far'], ordered=True)
    line_df = line_df.sort_values(['display_name', 'stratum_value'])
    for model_name in [model_labels[m] for m in model_order]:
        work = line_df[line_df['display_name'] == model_name]
        if work.empty:
            continue
        ax_line.plot(work['stratum_value'], work['pearson'], marker='o', linewidth=2.2, markersize=5.5, color=model_colors[model_name], label=model_name)
    ax_line.set_title('Train-distance trend', fontsize=11, pad=6)
    ax_line.set_xlabel('')
    ax_line.set_ylabel('Pearson')
    ax_line.set_ylim(0, 1.02)
    ax_line.grid(axis='y', alpha=0.22)
    ax_line.text(-0.24, 1.08, 'c', transform=ax_line.transAxes, fontsize=18, fontweight='bold')

    handles = [plt.Line2D([0], [0], color=model_colors[model_labels[m]], marker='s', lw=0, markersize=10, markeredgecolor='black', markeredgewidth=0.35) for m in model_order]
    labels = [model_labels[m] for m in model_order]
    fig.legend(handles, labels, frameon=False, ncol=5, loc='upper center', bbox_to_anchor=(0.5, 0.988))
    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.95])
    fig.savefig(combined_out, bbox_inches='tight')
    plt.close(fig)
    print(f'Exported {combined_out.name}')

    fig_line, ax = plt.subplots(1, 1, figsize=(5.1, 4.4), dpi=220)
    for model_name in [model_labels[m] for m in model_order]:
        work = line_df[line_df['display_name'] == model_name]
        if work.empty:
            continue
        ax.plot(work['stratum_value'], work['pearson'], marker='o', linewidth=2.4, markersize=6.0, color=model_colors[model_name], label=model_name)
    ax.set_title('Adamson train-distance trend')
    ax.set_xlabel('Train-distance bin')
    ax.set_ylabel('Pearson')
    ax.set_ylim(0, 1.02)
    ax.grid(axis='y', alpha=0.22)
    ax.legend(frameon=False, ncol=1, loc='center left', bbox_to_anchor=(1.02, 0.5))
    fig_line.tight_layout()
    fig_line.savefig(line_out, bbox_inches='tight')
    plt.close(fig_line)
    print(f'Exported {line_out.name}')


print(OUT_ROOT)
